In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\Arjun\OneDrive\Desktop\plfs_wage_analysis\PLFS-Wage-Analysis\cperv12025.csv")

# Keeping only working-age adults (15 to 59)
df_clean = df[(df['age'] >= 15) & (df['age'] <= 59)].copy()
print("Rows after age filter:", len(df_clean))

# Dropping missing and negative/zero wages
df_clean = df_clean.dropna(subset=['ern_reg']) #ern_reg -->Earnings For Regular Salaried/Wage Activity
df_clean = df_clean[df_clean['ern_reg'] > 0].copy()
# Renaming 'ern_reg' to 'wage' 
df_clean = df_clean.rename(columns={'ern_reg': 'wage'})
print("Rows after wage filter:", len(df_clean))

# Creating a dictionary mapping the MoSPI codes to actual years of schooling
edu_mapping = {
    1: 0, 2: 0, 3: 0, 4: 0, 
    5: 3, 
    6: 5, 
    7: 8, 
    8: 10, 
    10: 12, 
    11: 14, 
    12: 15, 
    13: 17
}

# Apply the mapping to create our new column
df_clean['schooling_years'] = df_clean['gedu_lvl'].map(edu_mapping)

# Drop anyone whose education code was missing/didn't map correctly
df_clean = df_clean.dropna(subset=['schooling_years'])

print("Rows after dropping missing education:", len(df_clean))

# Calculating Potential Experience (Age - Schooling - 6)
df_clean['exper'] = df_clean['age'] - df_clean['schooling_years'] - 6

# If someone started working early, the formula might give a negative number. We cap it at 0.
df_clean.loc[df_clean['exper'] < 0, 'exper'] = 0

# Create the true weight column (PLFS multiplier divided by 100)
df_clean['weight'] = df_clean['mult'] / 100

# Create the log wage column
df_clean['lnw'] = np.log(df_clean['wage'])

# Let's peek at the final variables to make sure everything calculated correctly
print(df_clean[['age', 'schooling_years', 'exper', 'wage', 'lnw', 'weight']].head())

df_clean.to_csv('clean_data.csv', index=False)

Rows after age filter: 763150
Rows after wage filter: 120326
Rows after dropping missing education: 120326
    age  schooling_years  exper   wage       lnw  weight
38   45                5     34   8000  8.987197  948.75
43   29               10     13  10000  9.210340  948.75
44   33               12     15  15000  9.615805  948.75
62   28               15      7  10000  9.210340  558.71
76   27               15      6  12000  9.392662  558.71
